# Front-end seed study -- **Account 2 of 2**

Learned filter at three seeds, plus one eq1 reference seed.

| # | front end | seed | est. |
|---|---|---|---|
| 1 | `learned` (constrained, N=4) | seed 42 | ~50 min |
| 2 | `eq1` (the paper's filter) | seed 43 | ~50 min |
| 3 | `learned` (constrained, N=4) | seed 43 | ~50 min |
| 4 | `learned` (constrained, N=4) | seed 44 | ~50 min |

Total ~3.3 h. **Runtime > Change runtime type > T4 GPU**, then Run All.

Ordered so the essentials finish first: after the first job on each account you
have `pyramid` and `learned` at 10,000 iterations, which is what the report
needs to be internally consistent. Everything after that buys statistical power.

This is a fresh account, so the first run downloads the 2.3 GB face crops and
asks for `kaggle.json`. Free Colab may reclaim the session; that is fine --
checkpoints go to Drive, finished jobs are skipped on a re-run, and an
interrupted one resumes where it stopped. Just Run All again.

`eq1` seed 42 already exists from the earlier round, so the two `eq1` jobs here
complete three seeds for the paper's own filter -- the reference distribution
the pyramid and learned variants get compared against.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Code and dependencies

In [ ]:
REPO = "https://github.com/rahulbhatyogesh/DIP-Project.git"

import os

if os.path.exists("/content/code"):
    !cd /content/code && git pull -q
else:
    !git clone -q {REPO} /content/code
%cd /content/code

!pip install -q timm albumentations 2>&1 | tail -2
import timm, albumentations
print("timm", timm.__version__, "| albumentations", albumentations.__version__)

In [ ]:
from src.models.build import verify_param_counts
assert verify_param_counts(), "parameter counts do not match the paper"
print("param counts match Table IV")

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
RUNS = "/content/drive/MyDrive/dip-hfc-runs"
os.makedirs(RUNS, exist_ok=True)
print("checkpoints ->", RUNS)

## 3. Face crops

Upload the **rahulbhat05** `kaggle.json` when prompted -- the dataset is private to that Kaggle account, and your Colab identity is unrelated to it. This takes a few minutes on a fresh account.

In [ ]:
DATASET = "rahulbhat05/celebdf-hfc-faces/versions/1"

from pathlib import Path
import json, os

if not Path("/content/faces/manifest.csv").exists():
    # kagglehub reads credentials from ~/.kaggle/kaggle.json or the environment.
    # With none it calls the API anonymously, and a PRIVATE dataset answers that
    # with 403 Forbidden -- not a login prompt.  Use the token of the Kaggle
    # account that OWNS the dataset (rahulbhat05), on every Colab account.
    cred = Path.home() / ".kaggle" / "kaggle.json"
    if not cred.exists():
        from google.colab import files
        print("Upload kaggle.json  (kaggle.com > Settings > Create New Token)")
        up = files.upload()
        assert up, "no file uploaded -- re-run this cell"
        cred.parent.mkdir(parents=True, exist_ok=True)
        cred.write_bytes(up[next(iter(up))])
        cred.chmod(0o600)
    tok = json.loads(cred.read_text())
    os.environ["KAGGLE_USERNAME"] = tok["username"]
    os.environ["KAGGLE_KEY"] = tok["key"]
    print("kaggle auth as:", tok["username"])

    !pip install -q kagglehub
    import kagglehub
    src = Path(kagglehub.dataset_download(DATASET))
    print("downloaded to:", src)
    faces = next(src.rglob("manifest.csv")).parent
    if not Path("/content/faces").exists():
        os.symlink(faces, "/content/faces")

assert Path("/content/faces/manifest.csv").exists(), "crops not found"
assert Path("/content/faces/splits.json").exists(), "splits.json missing"
!echo "crops: $(find -L /content/faces -name '*.jpg' | wc -l)"

## 4. Train and evaluate

Safe to re-run at any point -- finished jobs are skipped and an interrupted one resumes.

In [ ]:
import subprocess, time
from pathlib import Path

RUNS, FACES = "/content/drive/MyDrive/dip-hfc-runs", "/content/faces"
RES = RUNS + "/results10k"
Path(RES).mkdir(parents=True, exist_ok=True)


def stream(cmd):
    """Forward the child's output into the cell, and kill it on interrupt."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
    except BaseException:
        p.terminate()
        try:
            p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            p.kill()
        raise
    return p.wait()


JOBS = [('learned', 42), ('eq1', 43), ('learned', 43), ('learned', 44)]
EXTRA = {"pyramid": ["--pyramid-levels", "4"],
         "learned": ["--n-filters", "4"], "eq1": []}
NAME = {"pyramid": "pyramid", "learned": "learned", "eq1": "hfc"}

for n, (filt, seed) in enumerate(JOBS, 1):
    run = "xception_%s_identity_s%d_i10k" % (NAME[filt], seed)
    out, res = RUNS + "/" + run, RES + "/" + run + ".json"
    print("\n" + "=" * 70)
    print("[%d/%d] %s" % (n, len(JOBS), run))
    print("=" * 70, flush=True)

    if Path(res).exists():
        print("  already complete -- skipping")
        continue

    t0 = time.time()
    if not Path(out + "/summary.json").exists():
        rc = stream(["python", "-u", "-m", "src.train",
            "--faces-root", FACES, "--manifest", FACES + "/manifest.csv",
            "--splits", FACES + "/splits.json", "--backbone", "xception",
            "--mode", "hfc", "--hfc-filter", filt] + EXTRA[filt] + [
            "--protocol", "identity", "--seed", str(seed),
            "--max-iters", "10000", "--select-by", "auc", "--workers", "2",
            "--out", out, "--resume"])
        if rc != 0:
            print("  !! exited %d -- re-run this cell to resume" % rc)
            break
    else:
        print("  training already finished -- evaluating")

    stream(["python", "-u", "-m", "src.eval", "--ckpt", out + "/best.pt",
        "--faces-root", FACES, "--manifest", FACES + "/manifest.csv",
        "--splits", FACES + "/splits.json", "--part", "test", "--out", res])
    print("  done in %.1f min" % ((time.time() - t0) / 60))

print("\nall jobs for this account attempted")

## 5. Collect

Paste this output back into the chat.

In [ ]:
import json
from pathlib import Path

RES = Path("/content/drive/MyDrive/dip-hfc-runs/results10k")
print("#" * 70)
print("# ACCOUNT 2 -- copy everything below")
print("#" * 70)
for p in sorted(RES.glob("*_i10k.json")):
    d = json.load(open(p))
    f = d["frame"]
    print("\n%-44s bal_acc=%.4f auc=%.4f acc=%.4f iter=%s"
          % (p.stem, f["balanced_accuracy"], f["auc"], f["accuracy"],
             d.get("iteration")))